In [1]:
# Adobe Semantic Retrieval Pipeline

import os
import json
import time
from datetime import datetime
from typing import List, Dict
import fitz
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [2]:
from sentence_transformers import SentenceTransformer

# Load from HuggingFace
model = SentenceTransformer('all-MiniLM-L6-v2')
model.save("sentence_transformer_models/all-MiniLM-L6-v2")

In [3]:
## Main Work Starts

In [4]:
# Load locally saved model
MODEL_PATH = "sentence_transformer_models/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_PATH, device="cpu")

In [5]:
def extract_chunks_from_pdf(file_path: str) -> List[Dict]:
    doc = fitz.open(file_path)
    chunks = []

    for page_number, page in enumerate(doc, start=1):
        blocks = page.get_text("blocks")
        
        for block in blocks:
            if len(block) < 5:
                continue  # Malformed block
            
            text = block[4].strip()
            if text and len(text.split()) >= 10:  # 10 words or more
                chunks.append({
                    "text": text,
                    "page_number": page_number,
                    "document": os.path.basename(file_path)
                })

    doc.close()
    return chunks

In [6]:
# Create FAISS index
def build_faiss_index(chunks: List[Dict]) -> (faiss.IndexFlatL2, List[Dict]):
    embeddings = model.encode([c['text'] for c in chunks], show_progress_bar=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings, dtype='float32'))
    for i, emb in enumerate(embeddings):
        chunks[i]['embedding'] = emb  # Store for JSON if needed
    return index, chunks

In [7]:
# Perform semantic search
def semantic_search(query: str, index, chunks, top_k=5):
    query_embedding = model.encode([query])
    D, I = index.search(np.array(query_embedding, dtype='float32'), top_k)
    return [chunks[idx] for idx in I[0]]

In [12]:
# Format result to final JSON schema
def build_output_json(input_docs, persona, task, top_chunks):
    now = datetime.now().isoformat()
    return {
        "metadata": {
            "input_documents": input_docs,
            "persona": persona,
            "job_to_be_done": task,
            "processing_timestamp": now
        },
        "extracted_sections": [
            {
                "document": c['document'],
                "page_number": c['page_number'],
                "section_title": c['text'][:100],
                "importance_rank": i + 1
            }
            for i, c in enumerate(top_chunks)
        ],
        "subsection_analysis": [
            {
                "document": c['document'],
                "refined_text": c['text'],
                "page_number": c['page_number']
            }
            for c in top_chunks
        ]
    }


In [9]:
# Main pipeline runner
def run_adobe_pipeline(folder_paths: List[str], persona: str, task: str, top_k: int = 5) -> Dict:
    all_chunks = []
    input_files = []

    for folder in folder_paths:
        for fname in os.listdir(folder):
            if fname.endswith(".pdf"):
                path = os.path.join(folder, fname)
                chunks = extract_chunks_from_pdf(path)
                all_chunks.extend(chunks)
                input_files.append(fname)

    faiss_index, enriched_chunks = build_faiss_index(all_chunks)

    query = f"{persona} wants to: {task}"
    top_chunks = semantic_search(query, faiss_index, enriched_chunks, top_k)

    result_json = build_output_json(input_files, persona, task, top_chunks)
    return result_json


In [13]:
# ==== Example Usage ====
if __name__ == "__main__":
    folders = ["Cook/", "Travel/", "HR/"]
    persona = "HR manager"
    task = "What are benifits of generative AI?"


    output = run_adobe_pipeline(folders, persona, task)

    with open("pipeline_output.json", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print("\n\u2705 Pipeline completed. Output saved to 'pipeline_output.json'")


Batches:   0%|          | 0/91 [00:00<?, ?it/s]


✅ Pipeline completed. Output saved to 'pipeline_output.json'
